# Hybrid training with PyTorch
**Quantum Machine Learning · Week 3 (Variational quantum classifiers) · Day 3**

Lesson page: https://sara-aminpour.com/tutorials/qml/w3d3.html

In [ ]:
# On Google Colab, this installs qiskit (about a minute). On your own computer, skip it and
# install the packages once instead: pip install qiskit==2.5.2 scikit-learn torch matplotlib
import sys
if "google.colab" in sys.modules:
    %pip install --quiet qiskit==2.5.2

**Hybrid training with PyTorch.** Wrap a circuit as a PyTorch layer: the forward pass runs circuits, the backward
pass runs the parameter-shift rule, and PyTorch's chain rule connects it to ordinary classical layers. Then ask
the honest question: which part of the hybrid is doing the work?

In [ ]:
import numpy as np
import torch
from torch import nn
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorEstimator

torch.set_num_threads(1)
LAYERS = 2
x, w = ParameterVector("x", 2), ParameterVector("w", 4 * LAYERS)
circuit = QuantumCircuit(2)
circuit.ry(x[0], 0)
circuit.ry(x[1], 1)
for l in range(LAYERS):
    circuit.ry(w[4 * l], 0)
    circuit.rz(w[4 * l + 1], 0)
    circuit.ry(w[4 * l + 2], 1)
    circuit.rz(w[4 * l + 3], 1)
    circuit.cx(0, 1)
observable, estimator = SparsePauliOp("ZI"), StatevectorEstimator()

def run(jobs):
    """jobs: list of (angles of shape (batch, 2), weights). Returns ⟨Z⟩ of qubit 1, shape (len(jobs), batch)."""
    rows = np.array([np.concatenate([W, a]) for A, W in jobs for a in A])
    evs = estimator.run([(circuit, observable, rows)]).result()[0].data.evs
    return evs.reshape(len(jobs), -1)

class QuantumLayer(torch.autograd.Function):
    """Forward: one circuit per point. Backward: the parameter-shift rule, for the weights AND the input angles."""

    @staticmethod
    def forward(ctx, angles, weights):
        ctx.save_for_backward(angles, weights)
        return torch.tensor(run([(angles.detach().numpy(), weights.detach().numpy())])[0], dtype=torch.float32)

    @staticmethod
    def backward(ctx, grad_out):
        A, W = (t.detach().numpy().astype(np.float64) for t in ctx.saved_tensors)
        jobs = []
        for k in range(len(W)):                                   # shift one weight, for every point at once
            for s in (np.pi / 2, -np.pi / 2):
                Wk = W.copy()
                Wk[k] += s
                jobs.append((A, Wk))
        for j in range(A.shape[1]):                               # shift one input angle, for every point at once
            for s in (np.pi / 2, -np.pi / 2):
                Aj = A.copy()
                Aj[:, j] += s
                jobs.append((Aj, W))
        V = run(jobs)
        d = (V[0::2] - V[1::2]) / 2                               # (weights + inputs, batch) partial derivatives
        g = grad_out.numpy()
        grad_w = d[: len(W)] @ g                                  # each weight is shared by the whole batch: sum
        grad_a = (d[len(W):] * g).T                               # each angle belongs to one point
        QuantumLayer.runs += len(A) * (1 + len(jobs))
        return torch.tensor(grad_a, dtype=torch.float32), torch.tensor(grad_w, dtype=torch.float32)

QuantumLayer.runs = 0

# check the backward pass against finite differences
A = torch.tensor([[0.3, -0.7], [1.1, 0.4]], requires_grad=True)
Wt = torch.tensor(np.random.default_rng(0).normal(size=4 * LAYERS), dtype=torch.float32, requires_grad=True)
QuantumLayer.apply(A, Wt).sum().backward()
eps = 1e-3
up, down = Wt.detach().numpy().astype(np.float64), Wt.detach().numpy().astype(np.float64)
up[0] += eps
down[0] -= eps
numeric = (run([(A.detach().numpy(), up)])[0].sum() - run([(A.detach().numpy(), down)])[0].sum()) / (2 * eps)
print(f"d(sum)/d w0: parameter shift {Wt.grad[0].item():.5f}, central finite difference {numeric:.5f}")
print("gradients for the input angles:\n", A.grad.numpy().round(4))

## A hybrid model for handwritten digits
64 pixels are too many for 2 qubits, so a classical layer learns to squeeze each image into two angles.
After the circuit, a classical scale turns ⟨Z⟩ into a logit (Tuesday's $a\langle Z\rangle + b$).

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

digits = load_digits()
mask = np.isin(digits.target, [3, 8])
X_all = digits.data[mask] / 16.0
y_all = (digits.target[mask] == 8).astype(np.float32)
X_tr, X_te, y_tr, y_te = train_test_split(X_all, y_all, test_size=0.3, random_state=0, stratify=y_all)
X_tr, X_te = torch.tensor(X_tr, dtype=torch.float32), torch.tensor(X_te, dtype=torch.float32)
y_tr, y_te = torch.tensor(y_tr), torch.tensor(y_te)

class Hybrid(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Linear(64, 2)                                  # classical: 64 pixels → 2 numbers
        self.weights = nn.Parameter(0.6 * torch.randn(4 * LAYERS))       # quantum: 8 rotation angles
        self.scale = nn.Linear(1, 1)                                     # classical: a⟨Z⟩ + b

    def forward(self, X):
        angles = (np.pi / 2) * torch.tanh(self.encoder(X))               # squash into [-π/2, π/2]
        z = QuantumLayer.apply(angles, self.weights)
        return self.scale(z.unsqueeze(1)).squeeze(1)                     # a logit: p = sigmoid(logit)

def fit(model, epochs=15, lr=0.05, seed=0, report=True):
    torch.manual_seed(seed)
    optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()
    for epoch in range(epochs):
        for batch in torch.randperm(len(X_tr)).split(32):              # mini-batches of 32 images
            optimizer.zero_grad()
            loss = loss_fn(model(X_tr[batch]), y_tr[batch])
            loss.backward()                                              # classical autograd + parameter shift
            optimizer.step()
        if report and epoch % 5 == 4:
            with torch.no_grad():
                acc = ((model(X_te) > 0).float() == y_te).float().mean().item()
            print(f"epoch {epoch + 1:2d}: last batch loss {loss.item():.3f}, test accuracy {acc:.3f}")
    with torch.no_grad():
        return ((model(X_te) > 0).float() == y_te).float().mean().item()

torch.manual_seed(0)
hybrid = Hybrid()
print("parameters: classical", sum(p.numel() for n, p in hybrid.named_parameters() if n != "weights"), "+ quantum", hybrid.weights.numel())

In [ ]:
QuantumLayer.runs = 0
acc_hybrid = fit(hybrid)
print(f"hybrid test accuracy: {acc_hybrid:.3f}")
print(f"circuits run while training: {QuantumLayer.runs:,}")

## Is the quantum layer pulling its weight?
Swap the circuit for a tiny classical stand-in with a similar number of parameters, and train it the same way.

In [ ]:
class Twin(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Linear(64, 2)
        self.middle = nn.Sequential(nn.Tanh(), nn.Linear(2, 2), nn.Tanh(), nn.Linear(2, 1))   # 9 parameters, like the circuit's 8 + scale

    def forward(self, X):
        return self.middle(self.encoder(X)).squeeze(1)

torch.manual_seed(0)
acc_twin = fit(Twin(), report=False)
print(f"classical twin test accuracy: {acc_twin:.3f}   (hybrid: {acc_hybrid:.3f})")

## Your turn: freeze the encoder
Keep the classical encoder at its random starting weights (`requires_grad_(False)`) and train only the circuit and the
scale. What test accuracy do you get? What does that tell you about where the hybrid's skill came from?

In [ ]:
torch.manual_seed(0)
frozen = Hybrid()
# TODO: freeze frozen.encoder's parameters, train with fit(frozen), and print the test accuracy

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
torch.manual_seed(0)
frozen = Hybrid()
frozen.encoder.requires_grad_(False)                       # a random, fixed 64 → 2 projection
acc_frozen = fit(frozen, report=False)
torch.manual_seed(0)
frozen_twin = Twin()
frozen_twin.encoder.requires_grad_(False)                  # the same experiment for the classical twin
acc_frozen_twin = fit(frozen_twin, report=False)
print(f"frozen encoder, quantum middle  : test accuracy {acc_frozen:.3f}")
print(f"frozen encoder, classical middle: test accuracy {acc_frozen_twin:.3f}")
print(f"trained encoder, either middle  : {acc_hybrid:.3f} and {acc_twin:.3f}")
print("The trained encoder does most of the work: it learns which two numbers separate 3s from 8s,")
print("and after that almost any small model, quantum or classical, finishes the job.")